[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremylongshore/claude-code-plugins-plus-skills/blob/main/tutorials/skills/05-skill-validation.ipynb)

# Skill Validation (Tutorial 5/5)

> **Updated 2026-05-17 to schema 3.6.0.** Earlier versions of this notebook shipped a self-contained validator that taught rules now out of date — see [the repo quality audit](https://github.com/jeremylongshore/claude-code-plugins-plus-skills/blob/main/000-docs/266-RA-AUDT-repo-quality-audit-2026-05-17.md) for what changed and why. This tutorial now points at the canonical validator + spec instead of duplicating them.

## What you'll learn

1. The 8 required marketplace-tier frontmatter fields
2. The 7 required body sections
3. How to run the canonical validator against your skill
4. How to interpret the score, grade, and error output
5. Where to find rubric details when you hit a finding you don't understand

## Canonical sources (the source of truth, not this notebook)

- **Spec:** [`000-docs/6767-b-SPEC-DR-STND-claude-skills-standard.md`](https://github.com/jeremylongshore/claude-code-plugins-plus-skills/blob/main/000-docs/6767-b-SPEC-DR-STND-claude-skills-standard.md) — full schema 3.6.0 with field definitions, rubric, examples
- **Validator:** [`scripts/validate-skills-schema.py`](https://github.com/jeremylongshore/claude-code-plugins-plus-skills/blob/main/scripts/validate-skills-schema.py) — what CI actually runs
- **Schema changelog:** [`000-docs/SCHEMA_CHANGELOG.md`](https://github.com/jeremylongshore/claude-code-plugins-plus-skills/blob/main/000-docs/SCHEMA_CHANGELOG.md) — non-negotiables + version history
- **Wiki spec page:** [SKILL.md Specification](https://github.com/jeremylongshore/claude-code-plugins-plus-skills/wiki/SKILL-md-Specification)

## 1. The 8 required marketplace-tier frontmatter fields

Anthropic's spec requires only 2 fields (`name`, `description`). The Intent Solutions marketplace tier sits ON TOP of that and requires 8 — every published skill ships all of them.

| Field | Type | Purpose |
|---|---|---|
| `name` | string (kebab-case, ≤64 chars) | Unique slug for invocation |
| `description` | string (≤400 chars) | Single-sentence purpose; include 'Use when...' and 'Trigger with...' phrases |
| `allowed-tools` | CSV string OR YAML list | Tools the skill may use. Bash MUST be scoped: `Bash(git:*)`, never bare `Bash` |
| `version` | string | Semver: `1.0.0` |
| `author` | string | `Name <email>` format |
| `license` | string | SPDX identifier: `MIT`, `Apache-2.0` |
| `compatibility` | string | Free-text platform compatibility (e.g., `Designed for Claude Code`). Replaces deprecated `compatible-with` as of schema 3.4.0 |
| `tags` | YAML list | Discovery tags: `[devops, ci, deployment]` |

Both CSV string AND YAML list are valid for `allowed-tools` — the validator accepts either. The CSV form is older; the YAML list is the spec-compliant modern form.

## 2. The 7 required body sections

Marketplace-tier skills must include these H2 sections, in this order:

1. `## Overview`
2. `## Prerequisites`
3. `## Instructions`
4. `## Output`
5. `## Error Handling`
6. `## Examples`
7. `## Resources`

Missing any of these will fail validation. Empty sections (header + nothing useful below) will warn but not block — you should fill them in regardless.

## 3. Running the validator

Two tiers matter:

- **`--standard`**: Anthropic-spec-only (name + description). Use for quick checks.
- **`--marketplace`**: full 8-field IS rubric. This is what CI runs on every PR.

The validator can be pointed at a single skill, a plugin dir, or the whole repo.

In [ ]:
# Clone the repo (Colab) — skip if you're running locally
!git clone --depth 1 https://github.com/jeremylongshore/claude-code-plugins-plus-skills.git
%cd claude-code-plugins-plus-skills

In [ ]:
# Validate a single high-scoring sample skill (expect A grade, score ~96-98)
!python3 scripts/validate-skills-schema.py --marketplace plugins/performance/alerting-rule-creator/

## 4. Reading the output

The validator emits a per-skill block ending with a score and grade. The rubric breakdown shows where points were lost:

```
GRADE: A (97/100)
  Progressive Disclosure   28/30
  Ease Of Use              22/25
  Utility                  17/20
  Spec Compliance          15/15
  Writing Style             9/10
  Modifiers                -0
```

Grade thresholds:

| Score | Grade | Meaning |
|---|---|---|
| 90–100 | A | Marketplace-ready, no concerns |
| 80–89 | B | Minor warnings; usually shippable |
| 70–79 | C | Real gaps; address before publishing |
| 60–69 | D | Structural issues; needs rework |
| <60 | F | Stub or fundamentally broken |

The marketplace tier requires C or better — D/F fails CI.

In [ ]:
# Run against your own skill (replace path)
# !python3 scripts/validate-skills-schema.py --marketplace path/to/your/skill/

## 5. Common findings

If validation fails, look here first:

| Finding | Cause | Fix |
|---|---|---|
| `Missing required field: 'tags'` | Frontmatter missing the `tags:` key | Add a YAML list: `tags: [topic1, topic2]` |
| `Missing required field: 'compatibility'` | Using deprecated `compatible-with` | Migrate via `python3 scripts/batch-remediate.py --migrate-compatible-with` |
| `allowed-tools: unscoped 'Bash' not allowed` | Bare `Bash` in allowed-tools | Scope it: `Bash(git:*)`, `Bash(npm:*)`, etc. |
| `Required section missing: '## Examples'` | One of the 7 H2 sections absent | Add the section header + actual examples |
| `body < 30 lines` (stub) | SKILL.md too short | Flesh out content, or it's a stub skill that should be removed |
| Token budget exceeded (>500 lines) | Too much inline content | Move detail into `references/` subdir, link from SKILL.md |

For the full rubric + per-rule explanation, see the spec doc linked at the top.

## You're done

You've finished the 5-tutorial skills track:

1. [What is a Skill?](01-what-is-skill.ipynb)
2. [Skill Anatomy](02-skill-anatomy.ipynb)
3. [Build Your First Skill](03-build-your-first-skill.ipynb)
4. [Advanced Patterns](04-advanced-patterns.ipynb)
5. **Skill Validation** ← you are here

Next: write your own skill, validate it, open a PR. The [CONTRIBUTING.md](https://github.com/jeremylongshore/claude-code-plugins-plus-skills/blob/main/CONTRIBUTING.md) at repo root has the submission flow.